# 07 — Inondation côtière avec CLIMADA

**Objectif** : explorer le module `CoastalFlood` de CLIMADA pour l'analyse du risque de submersion marine.

**Différences avec l'inondation fluviale (06)** :
- La submersion côtière est dominée par la **montée du niveau marin (SLR)** et les **ondes de tempête**
- La **subsidence** (affaissement du sol) amplifie le risque dans certaines zones
- Les périodes de retour sont liées aux marées de tempête, pas au débit fluvial

**Zone** : Pays-Bas (Rotterdam, Amsterdam) + côte atlantique française

| Paramètre | Options |
|-----------|---------|
| Scénario | `historical`, `45` (RCP 4.5), `85` (RCP 8.5) |
| Année cible | `hist`, `2030`, `2050`, `2080` |
| Subsidence | `wtsub` (avec), `nosub` (sans) |
| Percentile SLR | `5`, `50`, `95` |

---

**Sommaire** :
1. Chargement du hazard côtier Aqueduct
2. Impact de la subsidence
3. Scénarios de montée du niveau marin
4. Portefeuille côtier et impact
5. Comparaison avec le notebook 02

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point

from climada.entity import Exposures, ImpactFuncSet
from climada.engine import Impact

from climada_petals.hazard import CoastalFlood
from climada_petals.entity.impact_funcs.river_flood import ImpfRiverFlood

print("Imports OK")

## 1. Hazard côtier — Aqueduct via CLIMADA

`CoastalFlood.from_aqueduct_tif()` télécharge les rasters de submersion côtière Aqueduct.

Contrairement au flood fluvial, le flood côtier a un paramètre **percentile** pour la montée du niveau marin (5e, 50e, 95e percentile des projections IPCC).

In [ ]:
# --- 1a. Hazard historique Pays-Bas ---
# ATTENTION : télécharge des rasters au premier appel
# Décommenter pour lancer

# cf_hist = CoastalFlood.from_aqueduct_tif(
#     rcp='historical',
#     target_year='hist',
#     return_periods=[10, 25, 50, 100, 250, 500, 1000],
#     subsidence='nosub',
#     countries=['NLD'],
# )
# print(f"Hazard côtier historique NL : {cf_hist.size} événements")
# print(f"Centroids : {cf_hist.centroids.size}")
# print(f"Profondeur max : {cf_hist.intensity.max():.1f} m")

print("API CoastalFlood.from_aqueduct_tif() :")
print("  rcp         : 'historical', '45', '85'")
print("  target_year : 'hist', 2030, 2050, 2080")
print("  subsidence  : 'nosub', 'wtsub'")
print("  percentile  : '5', '50', '95' (SLR)")
print("  countries   : ISO3 codes (ex: 'NLD', 'FRA')")

## 2. Exposition côtière

Portefeuille d'actifs en zone côtière basse : Pays-Bas + côte atlantique française.

In [ ]:
# --- 2. Portefeuille côtier ---
assets_coastal = [
    # Pays-Bas
    {"name": "Port Rotterdam",        "value": 200e6, "country": "NLD"},
    {"name": "Aéroport Schiphol",     "value": 500e6, "country": "NLD"},
    {"name": "Datacenter Amsterdam",  "value": 80e6,  "country": "NLD"},
    {"name": "Industrie Dordrecht",   "value": 45e6,  "country": "NLD"},
    # France côte atlantique
    {"name": "Port La Rochelle",      "value": 30e6,  "country": "FRA"},
    {"name": "Casino Royan",          "value": 12e6,  "country": "FRA"},
    {"name": "Résidence Noirmoutier", "value": 8e6,   "country": "FRA"},
    {"name": "Ostréiculture Arcachon","value": 5e6,   "country": "FRA"},
]
coords_coastal = [
    (4.48, 51.92),    # Rotterdam
    (4.76, 52.31),    # Schiphol
    (4.89, 52.37),    # Amsterdam
    (4.67, 51.81),    # Dordrecht
    (-1.15, 46.16),   # La Rochelle
    (-1.03, 45.62),   # Royan
    (-2.24, 47.00),   # Noirmoutier
    (-1.17, 44.66),   # Arcachon
]

exp_gdf = gpd.GeoDataFrame(assets_coastal, geometry=[Point(*c) for c in coords_coastal])
exp_gdf["impf_CF"] = 1  # Impact function côtière (JRC Europe residential)

exposure_coastal = Exposures(exp_gdf)
exposure_coastal.check()

print(f"Portefeuille côtier : {len(exposure_coastal.gdf)} actifs")
print(f"Valeur totale : {exposure_coastal.gdf['value'].sum()/1e6:.0f} M€")
print(f"  NLD : {exposure_coastal.gdf[exposure_coastal.gdf['country']=='NLD']['value'].sum()/1e6:.0f} M€")
print(f"  FRA : {exposure_coastal.gdf[exposure_coastal.gdf['country']=='FRA']['value'].sum()/1e6:.0f} M€")

## 3. Scénarios : subsidence + SLR

L'intérêt principal du flood côtier est la combinaison de facteurs :
- **Subsidence** : affaissement naturel + pompage (très fort aux Pays-Bas, -2 à -5 mm/an)
- **SLR** : montée du niveau marin (projections IPCC : +0.3m à +1m en 2100)
- **Percentile SLR** : incertitude des projections (5e = optimiste, 95e = pessimiste)

In [ ]:
# --- 3. Matrice de scénarios (décommenter quand données disponibles) ---

# Fonction d'impact (JRC Europe résidentielle, même pour le côtier)
impf_coastal = ImpfRiverFlood.from_jrc_region_sector("Europe", "residential")
impf_coastal.haz_type = "CF"  # Adapter le type d'aléa
impf_set_coastal = ImpactFuncSet([impf_coastal])

# coastal_scenarios = [
#     # Historique
#     {"label": "Historique (no sub)", "rcp": "historical", "year": "hist",
#      "sub": "nosub", "pct": "50", "color": "#1565C0", "ls": "solid"},
#     {"label": "Historique (with sub)", "rcp": "historical", "year": "hist",
#      "sub": "wtsub", "pct": "50", "color": "#2196F3", "ls": "dashed"},
#     # RCP 4.5 2050
#     {"label": "RCP 4.5 — 2050 (p50)", "rcp": "45", "year": 2050,
#      "sub": "wtsub", "pct": "50", "color": "#FF9800", "ls": "solid"},
#     {"label": "RCP 4.5 — 2050 (p95)", "rcp": "45", "year": 2050,
#      "sub": "wtsub", "pct": "95", "color": "#FF9800", "ls": "dashed"},
#     # RCP 8.5 2050
#     {"label": "RCP 8.5 — 2050 (p50)", "rcp": "85", "year": 2050,
#      "sub": "wtsub", "pct": "50", "color": "#e63946", "ls": "solid"},
#     {"label": "RCP 8.5 — 2050 (p95)", "rcp": "85", "year": 2050,
#      "sub": "wtsub", "pct": "95", "color": "#e63946", "ls": "dashed"},
#     # RCP 8.5 2080 (long terme)
#     {"label": "RCP 8.5 — 2080 (p95)", "rcp": "85", "year": 2080,
#      "sub": "wtsub", "pct": "95", "color": "#9C27B0", "ls": "dotted"},
# ]
#
# results_coastal = {}
# for sc in coastal_scenarios:
#     cf = CoastalFlood.from_aqueduct_tif(
#         rcp=sc["rcp"], target_year=sc["year"],
#         return_periods=[10, 25, 50, 100, 250, 500, 1000],
#         subsidence=sc["sub"], percentile=sc["pct"],
#         countries=['NLD'],
#     )
#     imp = Impact()
#     imp.calc(exposures=exposure_coastal, impact_funcs=impf_set_coastal, hazard=cf)
#     results_coastal[sc["label"]] = {
#         "impact": imp, "color": sc["color"], "ls": sc["ls"]
#     }
#     print(f"{sc['label']:35s} → EAI = {imp.aai_agg/1e6:.2f} M€/an")
#
# # Visualisation
# fig, ax = plt.subplots(figsize=(12, 6))
# rp = np.array([10, 25, 50, 100, 250, 500])
# for label, res in results_coastal.items():
#     fc = res["impact"].calc_freq_curve(return_per=rp)
#     ax.plot(fc.return_per, fc.impact/1e6, 'o-', color=res["color"],
#             linestyle=res["ls"], linewidth=2, label=label)
# ax.set_xlabel("Période de retour (années)")
# ax.set_ylabel("Perte (M€)")
# ax.set_title("Submersion côtière Pays-Bas — Scénarios SLR + subsidence")
# ax.set_xscale("log")
# ax.legend(fontsize=8)
# ax.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

print("Matrice de scénarios côtiers prête.")
print("7 scénarios : historique (±sub) + RCP 4.5/8.5 × 2050 (p50/p95) + RCP 8.5 2080 p95")

## 4. Récapitulatif — Risque côtier

### Facteurs de risque côtier

```
Niveau marin actuel
    + Montée du niveau marin (SLR) ──→ f(scénario RCP, horizon, percentile)
    + Subsidence locale              ──→ f(géologie, pompage, urbanisation)
    + Onde de tempête                ──→ f(période de retour)
    ────────────────────────────────
    = Hauteur d'eau totale           ──→ × Vulnérabilité (JRC) × Valeur actif
    = Dommage
```

### Spécificités vs inondation fluviale
- La **subsidence** est un facteur majeur (Pays-Bas : -2 à -5 mm/an, Jakarta : -25 mm/an)
- L'**incertitude SLR** est très large (facteur 3 entre p5 et p95)
- Les **défenses côtières** (digues, Deltawerken) ne sont pas modélisées dans Aqueduct
- Le risque augmente **exponentiellement** avec le SLR (non-linéarité topographique)

### Prochaines étapes
- [ ] Télécharger les données Aqueduct côtières NLD + FRA
- [ ] Comparer les résultats avec le notebook 02
- [ ] Explorer l'impact de la subsidence sur le portefeuille NLD
- [ ] Tester les horizons 2030/2050/2080 pour visualiser l'accélération
- [ ] Intégrer les données Deltares/DIVA pour une analyse plus fine